# Unemployment rate - pull, clean, export

Indicator: unemployment rate, annual average of monthly (not seasonally adjusted) values.
Source: FRED. Geographies: Allegheny County, Pennsylvania, United States.

**API key:** set `FRED_API_KEY` as an environment variable - do not paste it into this notebook.

In [ ]:
import json
import os
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests

API_KEY = os.environ.get("FRED_API_KEY")
if not API_KEY:
    raise RuntimeError("FRED_API_KEY not set. Set it as an environment variable, do not hardcode it here.")

RAW_DIR = Path("../raw")
DATA_DIR = Path("../data")
RAW_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

today = date.today().isoformat()
print("Pipeline run date:", today)

In [ ]:
SERIES = {
    "Allegheny County": "PAALLE3URN",
    "Pennsylvania": "PAUR",
    "United States": "UNRATE",
}

YEARS = list(range(2014, 2024))  # edit as needed; FRED updates monthly

## 1. Pull

One request per geography - FRED returns the full monthly history for a series in one call, not
one call per year. Caches every raw response, dated, before any parsing.

In [ ]:
raw_responses = {}
monthly_rows = []

for geo_name, series_id in SERIES.items():
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {"series_id": series_id, "api_key": API_KEY, "file_type": "json"}
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        payload = resp.json()
    except Exception as exc:
        print(f"FAILED  geo={geo_name}  series={series_id}  error={exc}")
        continue

    raw_responses[geo_name] = payload

    for obs in payload.get("observations", []):
        year = int(obs["date"][:4])
        if year not in YEARS:
            continue
        value = None if obs["value"] == "." else float(obs["value"])
        monthly_rows.append({"year": year, "geography": geo_name, "value": value})

    time.sleep(0.2)

with open(RAW_DIR / f"unemployment_rate_raw_{today}.json", "w") as f:
    json.dump(raw_responses, f, indent=2)

monthly_df = pd.DataFrame(monthly_rows)
monthly_df.head()

## 2. Clean: annual average of monthly values

Standard way to compare county/state/national unemployment across years without a 12-point-per-year chart.

In [ ]:
annual_df = (
    monthly_df.dropna(subset=["value"])
    .groupby(["year", "geography"], as_index=False)["value"]
    .mean()
    .round(1)
)

expected = len(YEARS) * len(SERIES)
print(f"Expected {expected} (year, geography) rows, got {len(annual_df)}")
annual_df

## 3. Export

In [ ]:
wide = annual_df.pivot(index="year", columns="geography", values="value").reset_index()
series = wide.to_dict(orient="records")

payload = {
    "indicator": "Unemployment rate (%), annual average",
    "geographies": list(SERIES.keys()),
    "unit": "percent",
    "snapshot_as_of": today,
    "status": "live",
    "source": "Federal Reserve Economic Data (FRED), series " + ", ".join(SERIES.values()) + " (underlying data from the Bureau of Labor Statistics Local Area Unemployment Statistics program)",
    "series": series,
    "notes": "Annual average of monthly, not seasonally adjusted values.",
}

with open(DATA_DIR / "unemployment_rate.json", "w") as f:
    json.dump(payload, f, indent=2)

print("Wrote", DATA_DIR / "unemployment_rate.json")
payload